<a href="https://colab.research.google.com/github/matthew-ngzc/AI-Safety-Module/blob/main/Week_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Clone the repository

In [2]:
try:
    ! git clone https://github.com/cs612-smu/cs612-smu-2025 CS612_SMU
    HOME_DIR = "./CS612_SMU/week5/"
except:
    print('Already clone!!!')

fatal: destination path 'CS612_SMU' already exists and is not an empty directory.


Exercise 2: In this exercise, we experiment with a simple simple MIA attack.

In [ ]:
## CIFAR100 ##
import torch

from torch import nn
from torch.utils.data import TensorDataset, DataLoader

from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms

import random

class CIFAR100Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(256 * 4 * 4, 512)
        self.fc2 = nn.Linear(512, 100)
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = F.relu(self.conv3(x))
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def save_model(model, name):
    torch.save(model.state_dict(), name)

def load_model(model_class, name, *args):
    model = model_class(*args)
    model.load_state_dict(torch.load(name, map_location=device))
    return model

def train(model, dataloader, loss_fn, optimizer, device):
    size = len(dataloader.dataset)
    model.train()
    for batch, (x, y) in enumerate(dataloader):
        x, y = x.to(device), y.to(device)
        # Compute prediction error
        pred = model(x)
        loss = loss_fn(pred, y)
        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if batch % 100 == 0:
            loss_val, current = loss.item(), batch * len(x)
            print('loss: {:.4f} [{}/{}]'.format(loss_val, current, size))

def test(model, dataloader, loss_fn, device):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    loss, correct = 0.0, 0
    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.int).sum().item()
    loss /= num_batches
    correct /= size
    print('Test Error: \n Accuracy: {:.2f}%, Avg loss: {:.4f}'.format(100 * correct, loss))

def attack(model, dataloader, loss_fn, device):
    size = 10000
    num_batches = len(dataloader)
    model.eval()
    correct = 0
    with torch.no_grad():
        for batch, (x, y) in enumerate(dataloader):
            x, y = x.to(device), y.to(device)
            pred = model(x)
            #TODO: add one line t update variable correct to evaluate the accuacy of this simple MIA attack
            #Note that this MIA attack predicts a sample is a member iff the prediction is correct.
            correct += (pred.argmax(1) == y).type(torch.int).sum().item()

            if (batch + 1) * len(y) == size: break
    return correct / size * 100

train_kwargs = {'batch_size': 1000}
test_kwargs = {'batch_size': 1000}
transform = transforms.ToTensor()

train_dataset = datasets.CIFAR100('./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR100('./data', train=False, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, **train_kwargs)
test_loader = torch.utils.data.DataLoader(test_dataset, **test_kwargs)


model = CIFAR100Net().to(device)
model = load_model(CIFAR100Net, HOME_DIR + 'exercise2/cifar100.pt').to(device)

MIAAttackTrain = attack(model, train_loader, nn.CrossEntropyLoss(), device)
MIAAttackTest = 100 - attack(model, test_loader, nn.CrossEntropyLoss(), device)

print('Overall MIA accuracy: {:.2f}%\n'.format((MIAAttackTrain+MIAAttackTest)/2))
print('MIA accuracy on train data: {:.2f}%\n'.format(MIAAttackTrain))
print('MIA accuracy on test data: {:.2f}%\n'.format(MIAAttackTest))

Overall MIA accuracy: 76.67%

MIA accuracy on train data: 89.34%

MIA accuracy on test data: 64.01%



Exercise 4: In this exercise, you will try training with differential privacy.

In [ ]:
import torch

from torch import nn
from torch.utils.data import TensorDataset, DataLoader

from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms

import numpy as np
import math


class MNISTNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 10)
        self.fc2 = nn.Linear(10, 10)
        self.fc3 = nn.Linear(10, 10)
        self.fc4 = nn.Linear(10, 10)

    def forward(self, x):
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        x = F.relu(x)
        x = self.fc3(x)
        x = F.relu(x)
        x = self.fc4(x)
        output = x # cross entropy in pytorch already includes softmax
        return output


def save_model(model, name):
    torch.save(model.state_dict(), name)


def load_model(model_class, name, *args):
    model = model_class(*args)
    model.load_state_dict(torch.load(name, map_location=torch.device('cpu')))

    return model


def train(model, dataloader, loss_fn, optimizer, device, delta, epsilon):
    sigma = math.sqrt(2 * math.log(1.25 / delta)) / epsilon
    size = len(dataloader.dataset)
    model.train()

    for batch, (x, y) in enumerate(dataloader):
        x, y = x.to(device), y.to(device)

        # Compute prediction error
        pred = model(x)
        loss = loss_fn(pred, y)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()

        #The following adds the noise according to differential privacy;
        for name, param in model.named_parameters():
            param.grad.data += np.random.normal(loc=0.0, scale=sigma, size=param.grad.data.size()) / len(y)

        optimizer.step()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * len(x)
            print('loss: {:.4f} [{}/{}]'.format(loss, current, size))


def test(model, dataloader, loss_fn, device):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)

    model.eval()
    loss, correct = 0.0, 0

    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)

            pred = model(x)
            loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.int).sum().item()

    loss /= num_batches
    correct /= size
    print('Test Error: \n Accuracy: {:.2f}%, Avg loss: {:.4f}\n'.format(100 * correct, loss))

def compute_mentr(pred, y):
    pred, y = pred.numpy(), y.numpy()
    mentr = []

    for i in range(len(y)):
        val = 0.0
        for j in range(len(pred[i])):
            if j == y[i]:
                val -= (1 - pred[i][j]) * math.log(pred[i][j])
            elif pred[i][j] < 1:
                val -= pred[i][j] * math.log(1 - pred[i][j])
        mentr.append(val)

    return np.array(mentr)

def attack(model, dataloader, loss_fn, device, threshold):
    size = 10000
    num_batches = len(dataloader)

    model.eval()
    correct = 0

    with torch.no_grad():
        for batch, (x, y) in enumerate(dataloader):
            x, y = x.to(device), y.to(device)
            pred = F.softmax(model(x), 1)
            mentr = compute_mentr(pred, y)
            correct += (mentr < threshold).sum()
            if (batch + 1) * len(y) == size: break

    return correct / size * 100


device = 'cpu'
train_kwargs = {'batch_size': 100}
test_kwargs = {'batch_size': 1000}
transform = transforms.ToTensor()

train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, **train_kwargs)
test_loader = torch.utils.data.DataLoader(test_dataset, **test_kwargs)

model = MNISTNet().to(device)

optimizer = optim.SGD(model.parameters(), lr=0.1)
num_of_epochs = 20
delta, epsilon = 1e-3, 10

print("A program demonstrating training with differential privacy.")

for epoch in range(num_of_epochs):
   print('\n------------- Epoch {} -------------\n'.format(epoch))
   train(model, train_loader, nn.CrossEntropyLoss(), optimizer, device, delta, epsilon)
   test(model, test_loader, nn.CrossEntropyLoss(), device)

threshold = 0.9

MIAAttackTrain = attack(model, train_loader, nn.CrossEntropyLoss(), device, threshold)
MIAAttackTest = 100 - attack(model, test_loader, nn.CrossEntropyLoss(), device, threshold)

print('Overall MIA accuracy: {:.2f}%\n'.format((MIAAttackTrain+MIAAttackTest)/2))
print('MIA accuracy on train data: {:.2f}%\n'.format(MIAAttackTrain))
print('MIA accuracy on test data: {:.2f}%\n'.format(MIAAttackTest))


A program demonstrating training with differential privacy.

------------- Epoch 0 -------------

loss: 2.3164 [0/60000]


/tmp/ipykernel_729/2770562971.py:68: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  param.grad.data += np.random.normal(loc=0.0, scale=sigma, size=param.grad.data.size()) / len(y)


loss: 2.0669 [10000/60000]
loss: 1.5024 [20000/60000]
loss: 1.1804 [30000/60000]
loss: 0.7290 [40000/60000]
loss: 0.5514 [50000/60000]
Test Error: 
 Accuracy: 84.23%, Avg loss: 0.5232


------------- Epoch 1 -------------

loss: 0.4948 [0/60000]
loss: 0.5182 [10000/60000]
loss: 1.0835 [20000/60000]
loss: 0.3807 [30000/60000]
loss: 0.3847 [40000/60000]
loss: 0.3689 [50000/60000]
Test Error: 
 Accuracy: 89.93%, Avg loss: 0.3377


------------- Epoch 2 -------------

loss: 0.2010 [0/60000]
loss: 0.4364 [10000/60000]
loss: 0.4435 [20000/60000]
loss: 0.2889 [30000/60000]
loss: 0.3182 [40000/60000]
loss: 0.3062 [50000/60000]
Test Error: 
 Accuracy: 90.86%, Avg loss: 0.2977


------------- Epoch 3 -------------

loss: 0.1617 [0/60000]
loss: 0.4078 [10000/60000]
loss: 0.3453 [20000/60000]
loss: 0.2491 [30000/60000]
loss: 0.2710 [40000/60000]
loss: 0.3473 [50000/60000]
Test Error: 
 Accuracy: 92.00%, Avg loss: 0.2647


------------- Epoch 4 -------------

loss: 0.1512 [0/60000]
loss: 0.3389 [10

# Exercise 5: MNIST model extraction
- use 10% of the original test set for your queries
- performance comparison for:
  1. only label
  2. full confidence
   
Evaluate attack using fidelity (rate of agreement between extracted and original model) and test accuracy 

In [20]:
# Exercise 5 setup
import random

import numpy as np
import torch
import torch.nn.functional as F
import torch.optim as optim
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, Subset
from torchvision import datasets, transforms


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class MNISTNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 10)
        self.fc2 = nn.Linear(10, 10)
        self.fc3 = nn.Linear(10, 10)
        self.fc4 = nn.Linear(10, 10)

    def forward(self, x):
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        return self.fc4(x)


class StudentCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)


def load_model(model_class, name, device):
    model = model_class()
    model.load_state_dict(torch.load(name, map_location=device))
    return model.to(device)


def build_test_loader(batch_size_eval):
    transform = transforms.ToTensor()
    test_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)
    test_loader = DataLoader(test_dataset, batch_size=batch_size_eval, shuffle=False)
    return test_dataset, test_loader




In [21]:
# Victim-query and boundary-scoring functions
def victim_confidence_query(victim, x_batch, device):
    victim.eval()
    with torch.no_grad():
        logits = victim(x_batch.to(device))
        probs = F.softmax(logits, dim=1)
    return probs.cpu()


def victim_label_query(victim, x_batch, device):
    probs = victim_confidence_query(victim, x_batch, device)
    return probs.argmax(dim=1)


def model_margin_score(victim, x_batch, device):
    probs = victim_confidence_query(victim, x_batch, device)
    top2 = torch.topk(probs, k=2, dim=1).values
    margin = top2[:, 0] - top2[:, 1]
    return margin


def surrogate_disagreement_score(model, x_batch, device, n_aug=5, noise_std=0.1):
    model.eval()
    x_device = x_batch.to(device)
    votes = []
    with torch.no_grad():
        for _ in range(n_aug):
            noise = torch.randn_like(x_device) * noise_std
            x_noisy = torch.clamp(x_device + noise, 0.0, 1.0)
            pred = model(x_noisy).argmax(dim=1)
            votes.append(pred.cpu())

    votes = torch.stack(votes, dim=0)
    scores = []
    for col in range(votes.shape[1]):
        vote_count = torch.bincount(votes[:, col], minlength=10).float()
        majority = vote_count.max().item()
        scores.append(1.0 - (majority / votes.shape[0]))
    return torch.tensor(scores)



In [22]:
# Surrogate training and evaluation functions
def train_label_surrogate(model, x_tensor, y_tensor, device, epochs=20, lr=1e-3, batch_size=128):
    dataset = TensorDataset(x_tensor, y_tensor)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    model.train()
    for _ in range(epochs):
        for x_batch, y_batch in loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            logits = model(x_batch)
            loss = loss_fn(logits, y_batch)
            loss.backward()
            optimizer.step()


def train_confidence_surrogate(model, x_tensor, p_tensor, device, epochs=20, lr=1e-3, batch_size=128, alpha=0.8):
    hard_y = p_tensor.argmax(dim=1)
    dataset = TensorDataset(x_tensor, p_tensor, hard_y)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    ce_loss = nn.CrossEntropyLoss()
    kl_loss = nn.KLDivLoss(reduction='batchmean')

    model.train()
    for _ in range(epochs):
        for x_batch, p_batch, y_batch in loader:
            x_batch = x_batch.to(device)
            p_batch = p_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            logits = model(x_batch)
            loss_kl = kl_loss(F.log_softmax(logits, dim=1), p_batch)
            loss_ce = ce_loss(logits, y_batch)
            loss = alpha * loss_kl + (1.0 - alpha) * loss_ce
            loss.backward()
            optimizer.step()


def evaluate_test_accuracy(model, dataloader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x_batch, y_batch in dataloader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            pred = model(x_batch).argmax(dim=1)
            correct += (pred == y_batch).sum().item()
            total += y_batch.shape[0]
    return correct / total


def evaluate_fidelity(extracted_model, victim_model, dataloader, device):
    extracted_model.eval()
    victim_model.eval()
    agree, total = 0, 0
    with torch.no_grad():
        for x_batch, _ in dataloader:
            x_batch = x_batch.to(device)
            extracted_pred = extracted_model(x_batch).argmax(dim=1)
            victim_pred = victim_model(x_batch).argmax(dim=1)
            agree += (extracted_pred == victim_pred).sum().item()
            total += x_batch.shape[0]
    return agree / total



In [23]:
# Shared-query extraction loop (no ground-truth labels used for query selection)
def build_shared_query_indices(dataset_size, query_budget, seed=42):
    rng = random.Random(seed)
    indices = list(range(dataset_size))
    rng.shuffle(indices)
    return indices[:query_budget]


def run_extraction(
    mode,
    victim_model,
    test_dataset,
    query_indices,
    device,
    batch_size=256,
):
    assert mode in ['label', 'confidence']

    query_subset = Subset(test_dataset, query_indices)
    query_loader = DataLoader(query_subset, batch_size=batch_size, shuffle=False)

    queried_x = []
    queried_label = []
    queried_prob = []

    for x_batch, _ in query_loader:
        queried_x.append(x_batch)

        if mode == 'label':
            y_batch = victim_label_query(victim_model, x_batch, device)
            queried_label.append(y_batch)
        else:
            p_batch = victim_confidence_query(victim_model, x_batch, device)
            queried_prob.append(p_batch)

    x_tensor = torch.cat(queried_x, dim=0)

    surrogate = StudentCNN().to(device)

    if mode == 'label':
        y_tensor = torch.cat(queried_label, dim=0).long()
        train_label_surrogate(surrogate, x_tensor, y_tensor, device)
    else:
        p_tensor = torch.cat(queried_prob, dim=0)
        assert torch.allclose(p_tensor.sum(dim=1), torch.ones(p_tensor.size(0)), atol=1e-4)
        train_confidence_surrogate(surrogate, x_tensor, p_tensor, device)

    return surrogate, set(query_indices)




In [24]:
# Runtime setup, data, and victim model
set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

batch_size_eval = 1024 if torch.cuda.is_available() else 256

test_dataset, test_loader = build_test_loader(batch_size_eval)

query_fraction = 0.10
query_budget = max(1, int(len(test_dataset) * query_fraction))
expected_budget = max(1, int(0.10 * len(test_dataset)))
assert query_budget == expected_budget
print(f'Test set size: {len(test_dataset)}, query budget (10%): {query_budget}')

shared_query_indices = build_shared_query_indices(len(test_dataset), query_budget, seed=42)
print(f'Shared query size for both modes: {len(shared_query_indices)}')

if 'HOME_DIR' not in globals():
    HOME_DIR = './CS612_SMU/week5/'

victim_path = HOME_DIR + 'exercise5/mnist.pt'
victim_model = load_model(MNISTNet, victim_path, device)
victim_model.eval()



Using device: cuda
Test set size: 10000, query budget (10%): 1000
Shared query size for both modes: 1000


MNISTNet(
  (fc1): Linear(in_features=784, out_features=10, bias=True)
  (fc2): Linear(in_features=10, out_features=10, bias=True)
  (fc3): Linear(in_features=10, out_features=10, bias=True)
  (fc4): Linear(in_features=10, out_features=10, bias=True)
)

In [25]:
# Run label-only extraction (shared queried indices)
label_model, label_queries = run_extraction(
    mode='label',
    victim_model=victim_model,
    test_dataset=test_dataset,
    query_indices=shared_query_indices,
    device=device,
)
print(f'Label-only queried samples: {len(label_queries)}')



Label-only queried samples: 1000


In [26]:
# Run full-confidence extraction (same shared queried indices)
confidence_model, confidence_queries = run_extraction(
    mode='confidence',
    victim_model=victim_model,
    test_dataset=test_dataset,
    query_indices=shared_query_indices,
    device=device,
)
print(f'Full-confidence queried samples: {len(confidence_queries)}')



Full-confidence queried samples: 1000


In [27]:
# Evaluate and compare both extracted models (strict holdout)
assert label_queries == confidence_queries, 'Both modes must use the same queried indices.'
used_for_training = set(shared_query_indices)
holdout_indices = [i for i in range(len(test_dataset)) if i not in used_for_training]
assert len(holdout_indices) > 0, 'No holdout samples left for evaluation.'

holdout_subset = Subset(test_dataset, holdout_indices)
holdout_loader = DataLoader(holdout_subset, batch_size=batch_size_eval, shuffle=False)

label_acc = evaluate_test_accuracy(label_model, holdout_loader, device)
label_fid = evaluate_fidelity(label_model, victim_model, holdout_loader, device)

conf_acc = evaluate_test_accuracy(confidence_model, holdout_loader, device)
conf_fid = evaluate_fidelity(confidence_model, victim_model, holdout_loader, device)

print('\nExtraction results (strict holdout; same queried indices):')
print(f'Holdout size: {len(holdout_subset)} / {len(test_dataset)}')
print('{:<16} {:>12} {:>14} {:>14}'.format('Mode', 'Queries', 'Test Acc', 'Fidelity'))
print('-' * 60)
print('{:<16} {:>12} {:>13.2f}% {:>13.2f}%'.format('Label-only', len(label_queries), 100 * label_acc, 100 * label_fid))
print('{:<16} {:>12} {:>13.2f}% {:>13.2f}%'.format('Full-confidence', len(confidence_queries), 100 * conf_acc, 100 * conf_fid))




Extraction results (strict holdout; same queried indices):
Holdout size: 9000 / 10000
Mode                  Queries       Test Acc       Fidelity
------------------------------------------------------------
Label-only               1000         91.17%         90.96%
Full-confidence          1000         92.96%         93.22%


### Implementation Summary

#### Architecture
- Used a small CNN architecture ( 2 convolutional blocks + 2 fully connected layers)
  1. Conv2d(1,32,3,padding=1), ReLU, MaxPool2d(2,2)
  2. Conv2d(32,64,3,padding=1), ReLU, MaxPool2d(2,2)
  3. Flatten
  4. Linear(64*7*7,128), ReLU
  5. Linear(128,10)

#### Training
- query samples were chosen randomly (shuffle, then choose the first 10%)
- used Adam 1e-3 for stability

#### Evaluation Results
- removed the 10% of samples used to train

| Mode | Queries | Test Accuracy | Fidelity |
|---|---:|---:|---:|
| Label-only | 1000 | 91.17% | 90.96% |
| Full-confidence | 1000 | 92.96% | 93.22% |

- Full-confidence performs better than label-only on both metrics (+1.79 pp accuracy, +2.26 pp fidelity). This is expected because confidence vectors provide richer supervision than hard labels under the same query budget.
- The high accuracy indicates that model extraction was successful. With just 10% of the queries we can have a 90% accuracy for both


Here, I explore using active selection to make the query selection more effecient.

This version keeps the original implementation intact and adds a stronger extraction pipeline with:
- Query budget capped at **10% of test set**.
- **No use of true labels** for query selection.
- Active querying via surrogate uncertainty + random exploration.
- Longer training with Adam, weight decay, LR scheduler, and early stopping (small fixed validation split).



In [28]:
# v2 helpers: shared active indices + improved training

def _split_train_val_indices(n, val_fraction=0.1, min_val=64, max_val=256, seed=1234):
    if n < 2:
        return list(range(n)), []

    val_n = int(n * val_fraction)
    val_n = max(min_val, val_n)
    val_n = min(max_val, val_n)
    val_n = min(val_n, n - 1)

    g = torch.Generator()
    g.manual_seed(seed + n)
    perm = torch.randperm(n, generator=g).tolist()

    val_idx = perm[:val_n]
    train_idx = perm[val_n:]
    return train_idx, val_idx


def _train_label_surrogate_v2(
    model,
    x_tensor,
    y_tensor,
    device,
    max_epochs=40,
    min_epochs=8,
    patience=4,
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=128,
):
    train_idx, val_idx = _split_train_val_indices(len(x_tensor), seed=2025)

    train_ds = TensorDataset(x_tensor[train_idx], y_tensor[train_idx])
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    has_val = len(val_idx) > 0
    if has_val:
        val_ds = TensorDataset(x_tensor[val_idx], y_tensor[val_idx])
        val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    opt = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)
    loss_fn = nn.CrossEntropyLoss()

    best_state = None
    best_val = float('inf')
    wait = 0

    for epoch in range(max_epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()

        sch.step()

        if has_val:
            model.eval()
            val_loss = 0.0
            count = 0
            with torch.no_grad():
                for xb, yb in val_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    logits = model(xb)
                    val_loss += loss_fn(logits, yb).item() * xb.size(0)
                    count += xb.size(0)
            val_loss /= max(1, count)

            if val_loss < best_val - 1e-5:
                best_val = val_loss
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                wait = 0
            else:
                wait += 1

            if epoch + 1 >= min_epochs and wait >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)


def _train_conf_surrogate_v2(
    model,
    x_tensor,
    p_tensor,
    device,
    max_epochs=40,
    min_epochs=8,
    patience=4,
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=128,
    alpha=0.7,
    temperature=2.0,
):
    hard_y = p_tensor.argmax(dim=1)
    train_idx, val_idx = _split_train_val_indices(len(x_tensor), seed=3030)

    train_ds = TensorDataset(x_tensor[train_idx], p_tensor[train_idx], hard_y[train_idx])
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    has_val = len(val_idx) > 0
    if has_val:
        val_ds = TensorDataset(x_tensor[val_idx], p_tensor[val_idx], hard_y[val_idx])
        val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    opt = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)
    ce_loss = nn.CrossEntropyLoss()
    kl_loss = nn.KLDivLoss(reduction='batchmean')

    best_state = None
    best_val = float('inf')
    wait = 0

    for epoch in range(max_epochs):
        model.train()
        for xb, pb, yb in train_loader:
            xb, pb, yb = xb.to(device), pb.to(device), yb.to(device)

            opt.zero_grad()
            logits = model(xb)

            log_q = F.log_softmax(logits / temperature, dim=1)
            p_t = torch.clamp(pb, min=1e-8)
            loss_kl = kl_loss(log_q, p_t) * (temperature ** 2)
            loss_ce = ce_loss(logits, yb)
            loss = alpha * loss_kl + (1.0 - alpha) * loss_ce

            loss.backward()
            opt.step()

        sch.step()

        if has_val:
            model.eval()
            val_loss = 0.0
            count = 0
            with torch.no_grad():
                for xb, pb, yb in val_loader:
                    xb, pb, yb = xb.to(device), pb.to(device), yb.to(device)
                    logits = model(xb)
                    log_q = F.log_softmax(logits / temperature, dim=1)
                    p_t = torch.clamp(pb, min=1e-8)
                    l_kl = kl_loss(log_q, p_t) * (temperature ** 2)
                    l_ce = ce_loss(logits, yb)
                    l = alpha * l_kl + (1.0 - alpha) * l_ce
                    val_loss += l.item() * xb.size(0)
                    count += xb.size(0)
            val_loss /= max(1, count)

            if val_loss < best_val - 1e-5:
                best_val = val_loss
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                wait = 0
            else:
                wait += 1

            if epoch + 1 >= min_epochs and wait >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)


def _pick_active_indices(model, test_dataset, unqueried_indices, k, device, candidate_pool_size=2000, explore_fraction=0.25):
    if k <= 0 or len(unqueried_indices) == 0:
        return []

    cand_size = min(candidate_pool_size, len(unqueried_indices))
    candidates = random.sample(unqueried_indices, cand_size)
    candidate_x = torch.stack([test_dataset[i][0] for i in candidates])

    scores = surrogate_disagreement_score(model, candidate_x, device, n_aug=7, noise_std=0.1)
    ranked = torch.argsort(scores, descending=True).tolist()
    ranked_candidates = [candidates[i] for i in ranked]

    explore_k = min(max(1, int(k * explore_fraction)), max(0, k - 1)) if k > 1 else 0
    exploit_k = k - explore_k

    chosen = ranked_candidates[:exploit_k]
    remain_pool = [i for i in unqueried_indices if i not in set(chosen)]
    if explore_k > 0 and len(remain_pool) > 0:
        chosen += random.sample(remain_pool, min(explore_k, len(remain_pool)))

    seen = set()
    out = []
    for i in chosen:
        if i in seen:
            continue
        seen.add(i)
        out.append(i)
        if len(out) == k:
            break
    return out


def build_shared_active_query_indices_v2(
    victim_model,
    test_dataset,
    query_budget,
    device,
    seed_queries=150,
    rounds=8,
    candidate_pool_size=2000,
):
    """Build ONE shared query set using label-only compliant active selection.

    Uses only victim hard labels for training the scout surrogate; does not use true labels
    or victim confidence outputs for query acquisition.
    """
    queried = set(random.sample(range(len(test_dataset)), min(seed_queries, query_budget)))

    def _query_labels(indices):
        xs, ys = [], []
        loader = DataLoader(Subset(test_dataset, indices), batch_size=256, shuffle=False)
        for xb, _ in loader:
            yb = victim_label_query(victim_model, xb, device)
            xs.append(xb)
            ys.append(yb)
        return torch.cat(xs, dim=0), torch.cat(ys, dim=0).long()

    x_tensor, y_tensor = _query_labels(sorted(list(queried)))
    scout = StudentCNN().to(device)
    _train_label_surrogate_v2(scout, x_tensor, y_tensor, device)

    per_round = max(1, (query_budget - len(queried)) // max(1, rounds))

    while len(queried) < query_budget:
        remain = query_budget - len(queried)
        k = min(per_round, remain)
        unqueried = [i for i in range(len(test_dataset)) if i not in queried]

        selected = _pick_active_indices(
            scout,
            test_dataset,
            unqueried,
            k,
            device,
            candidate_pool_size=candidate_pool_size,
            explore_fraction=0.25,
        )

        if len(selected) == 0:
            selected = random.sample(unqueried, min(k, len(unqueried)))

        queried.update(selected)

        x_tensor, y_tensor = _query_labels(sorted(list(queried)))
        _train_label_surrogate_v2(scout, x_tensor, y_tensor, device)

    return sorted(list(queried))


def train_from_shared_indices_v2(mode, victim_model, test_dataset, query_indices, device):
    assert mode in ['label', 'confidence']

    xs = []
    ys = []
    ps = []

    loader = DataLoader(Subset(test_dataset, query_indices), batch_size=256, shuffle=False)
    for xb, _ in loader:
        xs.append(xb)
        if mode == 'label':
            ys.append(victim_label_query(victim_model, xb, device))
        else:
            ps.append(victim_confidence_query(victim_model, xb, device))

    x_tensor = torch.cat(xs, dim=0)
    model = StudentCNN().to(device)

    if mode == 'label':
        y_tensor = torch.cat(ys, dim=0).long()
        _train_label_surrogate_v2(model, x_tensor, y_tensor, device)
    else:
        p_tensor = torch.cat(ps, dim=0)
        assert torch.allclose(p_tensor.sum(dim=1), torch.ones(p_tensor.size(0)), atol=1e-4)
        _train_conf_surrogate_v2(model, x_tensor, p_tensor, device)

    return model




In [29]:
# v2 run: shared active queried indices for both modes
set_seed(2026)

v2_query_budget = max(1, int(len(test_dataset) * 0.10))
assert v2_query_budget <= int(len(test_dataset) * 0.10)

shared_query_indices_v2 = build_shared_active_query_indices_v2(
    victim_model=victim_model,
    test_dataset=test_dataset,
    query_budget=v2_query_budget,
    device=device,
)

label_model_v2 = train_from_shared_indices_v2(
    mode='label',
    victim_model=victim_model,
    test_dataset=test_dataset,
    query_indices=shared_query_indices_v2,
    device=device,
)

conf_model_v2 = train_from_shared_indices_v2(
    mode='confidence',
    victim_model=victim_model,
    test_dataset=test_dataset,
    query_indices=shared_query_indices_v2,
    device=device,
)

label_queries_v2 = set(shared_query_indices_v2)
conf_queries_v2 = set(shared_query_indices_v2)

print(f'v2 Shared queried samples: {len(shared_query_indices_v2)}')



v2 Shared queried samples: 1000


In [30]:
# v2 evaluation on strict holdout (same queried indices for both models)
assert label_queries_v2 == conf_queries_v2
used_for_training_v2 = set(shared_query_indices_v2)
holdout_indices_v2 = [i for i in range(len(test_dataset)) if i not in used_for_training_v2]
assert len(holdout_indices_v2) > 0

holdout_loader_v2 = DataLoader(Subset(test_dataset, holdout_indices_v2), batch_size=batch_size_eval, shuffle=False)

label_acc_v2 = evaluate_test_accuracy(label_model_v2, holdout_loader_v2, device)
label_fid_v2 = evaluate_fidelity(label_model_v2, victim_model, holdout_loader_v2, device)

conf_acc_v2 = evaluate_test_accuracy(conf_model_v2, holdout_loader_v2, device)
conf_fid_v2 = evaluate_fidelity(conf_model_v2, victim_model, holdout_loader_v2, device)

print('\nV2 extraction results (strict holdout, <=10% queries, same queried indices):')
print(f'Holdout size: {len(holdout_indices_v2)} / {len(test_dataset)}')
print('{:<16} {:>12} {:>14} {:>14}'.format('Mode', 'Queries', 'Test Acc', 'Fidelity'))
print('-' * 60)
print('{:<16} {:>12} {:>13.2f}% {:>13.2f}%'.format('Label-only v2', len(label_queries_v2), 100 * label_acc_v2, 100 * label_fid_v2))
print('{:<16} {:>12} {:>13.2f}% {:>13.2f}%'.format('Full-conf v2', len(conf_queries_v2), 100 * conf_acc_v2, 100 * conf_fid_v2))




V2 extraction results (strict holdout, <=10% queries, same queried indices):
Holdout size: 9000 / 10000
Mode                  Queries       Test Acc       Fidelity
------------------------------------------------------------
Label-only v2            1000         94.20%         94.44%
Full-conf v2             1000         94.58%         95.11%
